In [2]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [4]:
from langchain_core.documents import Document

In [5]:
sample_doc = Document(
    page_content = "hello",
    metadata = {"source":"https://www.google.com"}
)

In [6]:
sample_doc


Document(metadata={'source': 'https://www.google.com'}, page_content='hello')

In [7]:
type(sample_doc)

langchain_core.documents.base.Document

In [8]:
# text data

In [9]:
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/Python.txt",encoding="utf-8")

/var/folders/tq/0f_qj4xj7t10c1gmmkp7dkxw0000gn/T/ipykernel_51164/402599292.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [10]:
document = loader.load()

In [11]:
document

[Document(metadata={'source': 'data/Python.txt'}, page_content='\ufeffPython is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, 

In [12]:
# pdf data 

In [13]:
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research2.pdf")

# document = pdf_loader.load()

In [14]:
# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("data/research2.pdf")

# document = pdf_loader.load()


## Ingestion Pipeline

In [15]:
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

### Documents

In [16]:
def load_all_pdf():
    folder_path = "data/pdfs"
    num_doc = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path,filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_doc +=1
    print("total_pdfs:",num_doc)
    print("total_pages:",len(all_docs))
    return all_docs

In [17]:
all_pdf_doc = load_all_pdf()

total_pdfs: 1
total_pages: 21


### Chunks

In [18]:
# chunks 

# !pip install langchain_text_splitters

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_doc(documents,chunk_size=500,chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_doc = text_splitter.split_documents(documents)
    return chunked_doc

In [20]:
chunks = split_doc(all_pdf_doc)

In [21]:
len(chunks)

244

### Embeddings

In [22]:
from sentence_transformers import SentenceTransformer

In [23]:
class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"):

        self.model_name=model_name
        print("loading model....",self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=",self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self,text):
        embeddings = self.model.encode(text,show_progress_bar=True)
        print("embeddings shape:",embeddings.shape)
        return embeddings

In [30]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


/var/folders/tq/0f_qj4xj7t10c1gmmkp7dkxw0000gn/T/ipykernel_51164/2522851818.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=",self.model.get_sentence_embedding_dimension())


### Vector Store

In [31]:
import chromadb
import uuid


In [32]:
class VectorStoreManager:
    def __init__(self,persist_directory="data/vector_store",collection_name="pdf_doc"): # persist_directory - where we want to store the embeddings, collection_name - name of the collection of all the embeddings
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self.initialize_store()

    def initialize_store(self):
        os.makedirs(self.persist_directory,exist_ok=True)
        # create client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata = {"description":"vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection",self.collection_name)
        print("docs in collection:",self.collection.count())

    
    def add_documents(self, documents, embeddings):
    
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents doesn't match number of embeddings")
    
        ids = []
        all_metadata = []
        doc_content = []
        embeddings_list = []
    
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
    
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)
    
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
    
            all_metadata.append(metadata)
            doc_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
    
        # Add all documents to ChromaDB in one call
        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=doc_content,
            embeddings=embeddings_list
        )
    
        print("Total documents added:", len(doc_content))
        print("Documents in collection:", self.collection.count())

        
    

In [33]:
vector_store = VectorStoreManager()

initialized the vector store with collection pdf_doc
docs in collection: 244


In [34]:
# data => docs => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks,embedding)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

embeddings shape: (244, 384)
total docs added in vector store: 244
docs in collection: 488


## Retrieval Pipeline

In [35]:
from sklearn.metrics.pairwise import cosine_similarity


In [36]:
class RAGRetriever:
    def __init__(self,embedding_manager,vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self,query,top_k=5,score_threshold=0.0):
        # query => embeddings
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        # cosine similarity
        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0] # cosine distances

            for i , (ids,metadata,document,distance) in enumerate(zip(ids,metadatas,documents,distances)):
                similarity_score = 1-distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id":ids,
                        "document":document,
                        "metadata":metadata,
                        "distance":distance,
                        "similarity_score ":similarity_score, 
                        "rank": i+1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no docs found")

        return retrieved_docs
            

In [37]:
rag_retriever = RAGRetriever(embedding_manager,vector_store)

In [38]:
rag_retriever.retrieve("what is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_0141a01d-53e8-453c-b664-449fc42b967a',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'keywords': '',
   'author': '',
   'title': '',
   'total_pages': 21,
   'page_label': '1',
   'producer': 'pdfTeX-1.40.25',
   'page': 0,
   'source': 'data/pdfs/research2.pdf',
   'creator': 'LaTeX with hyperref',
   'subject': '',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'doc_index': 11,
   'moddate': '2024-03-28T00:54:45+00:00',
   'content_length': 288,
   'trapped': '/False'},
  'distance': 0.46291521191596985,
  'similarity_score ': 0.5370847880840302,
  'rank': 1},
 {'id': 'doc_33801862

## Integrate with LLMs

In [76]:
# OpenAI - GPT 

In [39]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [40]:
from dotenv import load_dotenv

# load variables from the env file
load_dotenv()

# read the api key
api_key = os.getenv("OPENAI_API_KEY")

print(api_key is not None)

True


In [41]:
# !pip install langchain-openai

In [53]:
# %pip install --upgrade openai httpx httpcore brotli

In [54]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key = api_key,
    model = "gpt-5.4",
    temperature= 0.1, # temp -> low -> fact based , temp -> high -> creative 
    max_tokens=1024 # number of tokens in the output
)

In [62]:
# generate our retrieval-augmented output

def generate_output(query,rag_retriever,llm,top_k=3):
    results = rag_retriever.retrieve(query,top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("no relevant context found for the given query")
        
    # query + context
    prompt = f""" use the context to generate the answer for the query 
                    context : {context}
                    query : {query}"""
    
    response = llm.invoke(prompt) # expecting a string as prompt 
    return response.content

In [1]:
answer = generate_output("what is RAG?",rag_retriever,llm)

## Groq

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

print("API key loaded:", groq_api_key is not None)

In [67]:
!pip install langchain-groq

In [75]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    api_key = groq_api_key ,
    model = "openai/gpt-oss-120b",
    temperature= 0.1, # temp -> low -> fact based , temp -> high -> creative 
    max_tokens=1024 # number of tokens in the output
)

In [76]:
# generate our retrieval-augmented output

def generate_output(query,rag_retriever,llm,top_k=3):
    results = rag_retriever.retrieve(query,top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("no relevant context found for the given query")
        
    # query + context
    prompt = f""" use the context to generate the answer for the query 
                    context : {context}
                    query : {query}"""
    
    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt 
    return response.content

In [77]:
answer = generate_output("what is RAG?",rag_retriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [78]:
print(answer)

**RAG (Retrieval‑Augmented Generation)** is a hybrid architecture that combines two complementary capabilities:

| Component | What it does | Why it matters |
|-----------|--------------|----------------|
| **Retriever** | Searches an external knowledge source (e.g., a document corpus, a vector database, a search engine) to pull the most relevant pieces of information for a given query. | Provides up‑to‑date, factual, and domain‑specific context that a language model alone may not know. |
| **Generator** | A large language model (LLM) that takes the retrieved passages as additional input and produces a natural‑language response. | Allows the model to ground its output in concrete evidence, improving accuracy, factuality, and controllability. |

In short, **RAG = Retrieval + Generation**. The system first *retrieves* relevant text, then *generates* an answer that weaves that text together with the model’s own reasoning abilities.

---

### How a typical RAG pipeline works

1. **Query en